<a href="https://www.kaggle.com/code/harshit1631/testing-on-official-testset?scriptVersionId=304303409" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import os

DATASET_PATH = "/kaggle/input"

for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

    subindent = " " * 2 * (level + 1)
    for f in files[:5]:  # show only first 5 files per folder
        print(f"{subindent}{f}")

In [ ]:
import os

print(os.listdir("/kaggle/input"))

In [ ]:
import pandas as pd

csv_path = "/kaggle/input/datasets/himanshurajput123/testing/ukdd_navi_00051_00068_00076/ukdd_navi_00051.csv"

df = pd.read_csv(csv_path)

print(df.head())
print(df.columns)
print(len(df))

In [ ]:
import pandas as pd

csv_path = "/kaggle/input/datasets/himanshurajput123/testing/ukdd_navi_00051_00068_00076/ukdd_navi_00051.csv"

df = pd.read_csv(csv_path, sep=";")

print(df.head())
print(df.columns)
print(len(df))

In [ ]:
import torch

path = "/kaggle/input/models/harshit1631/all-models-till-now/pytorch/default/1/final_model.pth"

obj = torch.load(path, map_location="cpu")

print(type(obj))

if isinstance(obj, dict):
    print(obj.keys())

In [ ]:
import torch

path = "/kaggle/input/models/harshit1631/all-models-till-now/pytorch/default/1/model_0.pth"

obj = torch.load(path, map_location="cpu")

print(type(obj))

if isinstance(obj, dict):
    print(obj.keys())

In [ ]:
import torch

path = "/kaggle/input/models/harshit1631/all-models-till-now/pytorch/default/1/model_0.pth"

sd = torch.load(path, map_location="cpu")

print(sd["classifier.1.weight"].shape)

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b3

In [ ]:
def build_model(num_classes=15):

    model = efficientnet_b3(weights=None)

    in_features = model.classifier[1].in_features

    model.classifier[1] = nn.Linear(in_features, num_classes)

    return model

In [ ]:
MODEL_PATH = "/kaggle/input/models/harshit1631/all-models-till-now/pytorch/default/1/model_0.pth"

model = build_model(15)

sd = torch.load(MODEL_PATH, map_location="cpu")

model.load_state_dict(sd)

model.eval()

print("loaded")

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b3

def build_model(num_classes=15):

    model = efficientnet_b3(weights=None)

    in_features = model.classifier[1].in_features

    model.classifier[1] = nn.Linear(in_features, num_classes)

    return model

In [ ]:
PATH0 = "/kaggle/input/models/harshit1631/all-models-till-now/pytorch/default/1/model_0.pth"
PATH1 = "/kaggle/input/models/harshit1631/all-models-till-now/pytorch/default/1/model_1.pth"
PATH2 = "/kaggle/input/models/harshit1631/all-models-till-now/pytorch/default/1/model_2.pth"

model0 = build_model()
model1 = build_model()
model2 = build_model()

model0.load_state_dict(torch.load(PATH0, map_location="cpu"))
model1.load_state_dict(torch.load(PATH1, map_location="cpu"))
model2.load_state_dict(torch.load(PATH2, map_location="cpu"))

model0.cuda().eval()
model1.cuda().eval()
model2.cuda().eval()

print("ensemble loaded")

In [ ]:
import torch
import torch.nn as nn

from torchvision.models import (
    efficientnet_b3,
    resnet50,
    densenet121
)

In [ ]:
def build_eff():

    m = efficientnet_b3(weights=None)

    in_f = m.classifier[1].in_features

    m.classifier[1] = nn.Linear(in_f, 15)

    return m

In [ ]:
def build_res():

    m = resnet50(weights=None)

    in_f = m.fc.in_features

    m.fc = nn.Linear(in_f, 15)

    return m

In [ ]:
def build_dense():

    m = densenet121(weights=None)

    in_f = m.classifier.in_features

    m.classifier = nn.Linear(in_f, 15)

    return m

In [ ]:
PATH0 = "/kaggle/input/models/harshit1631/all-models-till-now/pytorch/default/1/model_0.pth"
PATH1 = "/kaggle/input/models/harshit1631/all-models-till-now/pytorch/default/1/model_1.pth"
PATH2 = "/kaggle/input/models/harshit1631/all-models-till-now/pytorch/default/1/model_2.pth"

model0 = build_eff()
model1 = build_res()
model2 = build_dense()

model0.load_state_dict(torch.load(PATH0, map_location="cpu"))
model1.load_state_dict(torch.load(PATH1, map_location="cpu"))
model2.load_state_dict(torch.load(PATH2, map_location="cpu"))

model0.cuda().eval()
model1.cuda().eval()
model2.cuda().eval()

print("ALL 3 LOADED")

In [ ]:
from PIL import Image
import os

img_path = "/kaggle/input/datasets/himanshurajput123/testing/Testdata_ICPR_2026_RARE_Challenge/ukdd_navi_00051"

files = os.listdir(img_path)

sample = os.path.join(img_path, files[0])

img = Image.open(sample)

print(img.size)

In [ ]:
import torch
from torchvision import transforms
from PIL import Image

transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
import torch.nn.functional as F

def predict_image(path):

    img = Image.open(path).convert("RGB")

    x = transform(img).unsqueeze(0).cuda()

    with torch.no_grad():

        o0 = model0(x)
        o1 = model1(x)
        o2 = model2(x)

        out = (o0 + o1 + o2) / 3

        prob = torch.sigmoid(out)

    return prob.cpu().numpy()[0]

In [ ]:
MODEL_LABELS = [
    "mouth",
    "esophagus",
    "stomach",
    "small intestine",
    "colon",
    "z-line",
    "pylorus",
    "ileocecal valve",
    "active bleeding",
    "angiectasia",
    "blood",
    "erosion",
    "erythema",
    "polyp",
    "ulcer",
]

In [ ]:
img = "/kaggle/input/datasets/himanshurajput123/testing/Testdata_ICPR_2026_RARE_Challenge/ukdd_navi_00051/frame_0000049.png"

p = predict_image(img)

print(p)

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

THRESH = 0.5


def predict_video(video_folder, save_csv):

    files = sorted(os.listdir(video_folder))

    rows = []

    for f in tqdm(files):

        path = os.path.join(video_folder, f)

        prob = predict_image(path)

        labels = (prob > THRESH).astype(int)

        row = {"frame_file": f}

        for i, name in enumerate(MODEL_LABELS):
            row[name] = int(labels[i])

        rows.append(row)

    df = pd.DataFrame(rows)

    df.to_csv(save_csv, index=False)

    print("saved", save_csv)

In [ ]:
video = "/kaggle/input/datasets/himanshurajput123/testing/Testdata_ICPR_2026_RARE_Challenge/ukdd_navi_00051"

predict_video(
    video,
    "/kaggle/working/ukdd_navi_00051.csv"
)

In [ ]:
predict_video(
    "/kaggle/input/datasets/himanshurajput123/testing/Testdata_ICPR_2026_RARE_Challenge/ukdd_navi_00068",
    "/kaggle/working/ukdd_navi_00068.csv"
)

predict_video(
    "/kaggle/input/datasets/himanshurajput123/testing/Testdata_ICPR_2026_RARE_Challenge/ukdd_navi_00076",
    "/kaggle/working/ukdd_navi_00076.csv"
)

In [ ]:
import torch

path = "PUT_PATH"

obj = torch.load(path, map_location="cpu")

print(type(obj))

if isinstance(obj, dict):
    print(list(obj.keys())[:20])

In [ ]:
import os

path = "/kaggle/input/models/harshit1631/body/pytorch/default/1"

print(os.listdir(path))

In [ ]:
import torch

path = "/kaggle/input/models/harshit1631/body/pytorch/default/1/best_anatomy_model"

obj = torch.load(path, map_location="cpu")

print(type(obj))

if isinstance(obj, dict):
    keys = list(obj.keys())
    print(keys[:20])

In [ ]:
import os

path = "/kaggle/input/models/harshit1631/body/pytorch/default/1/best_anatomy_model"

print(os.listdir(path))

In [ ]:
import torch

path = "/kaggle/input/models/harshit1631/body-model/pytorch/default/1/best_anatomy_model.pth"

obj = torch.load(path, map_location="cpu")

print(type(obj))

if isinstance(obj, dict):
    keys = list(obj.keys())
    print(keys[:20])

In [ ]:
import torch

path = "/kaggle/input/models/harshit1631/body-model/pytorch/default/1/best_anatomy_model.pth"

obj = torch.load(path, map_location="cpu")

sd = obj["model_state"]

print(list(sd.keys())[:20])

In [ ]:
import torch

path = "/kaggle/input/models/harshit1631/body-model/pytorch/default/1/best_anatomy_model.pth"

obj = torch.load(path, map_location="cpu")

print(obj["anatomy_cols"])


In [ ]:
!pip install timm

In [ ]:
import timm
import torch.nn as nn
import torch

ANATOMY_PATH = "/kaggle/input/models/harshit1631/body-model/pytorch/default/1/best_anatomy_model.pth"

ckpt = torch.load(ANATOMY_PATH, map_location="cpu")

anatomy_cols = ckpt["anatomy_cols"]

num_classes = len(anatomy_cols)

model_anat = timm.create_model(
    "efficientnet_b3",
    pretrained=False,
    num_classes=num_classes
)

model_anat.load_state_dict(ckpt["model_state"])

model_anat.cuda().eval()

print("anatomy loaded")

In [ ]:
import timm
import torch
import torch.nn as nn

ANATOMY_PATH = "/kaggle/input/models/harshit1631/body-model/pytorch/default/1/best_anatomy_model.pth"

ckpt = torch.load(ANATOMY_PATH, map_location="cpu")

anatomy_cols = ckpt["anatomy_cols"]

num_anatomy = len(anatomy_cols)
num_disease = 15   # from your main model

In [ ]:
class MultiHeadModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b4",
            pretrained=False,
            num_classes=0
        )

        in_features = self.backbone.num_features

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.shared = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512)
        )

        self.anatomy_head = nn.Linear(512, num_anatomy)

        self.disease_head = nn.Linear(512, num_disease)

    def forward(self, x):

        x = self.backbone.forward_features(x)

        x = self.pool(x).flatten(1)

        x = self.shared(x)

        anat = self.anatomy_head(x)

        dis = self.disease_head(x)

        return anat, dis

In [ ]:
model_anat = MultiHeadModel()

model_anat.load_state_dict(ckpt["model_state"])

model_anat.cuda().eval()

print("loaded OK")

In [ ]:
import timm
import torch
import torch.nn as nn

ANATOMY_PATH = "/kaggle/input/models/harshit1631/body-model/pytorch/default/1/best_anatomy_model.pth"

ckpt = torch.load(ANATOMY_PATH, map_location="cpu")

anatomy_cols = ckpt["anatomy_cols"]

num_anatomy = len(anatomy_cols)
num_disease = 15


class MultiHeadModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b4",
            pretrained=False,
            num_classes=0
        )

        in_features = self.backbone.num_features

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.shared = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512)
        )

        self.anatomy_head = nn.Linear(512, num_anatomy)

        self.disease_head = nn.Linear(512, num_disease)

    def forward(self, x):

        x = self.backbone.forward_features(x)

        x = self.pool(x).flatten(1)

        x = self.shared(x)

        anat = self.anatomy_head(x)
        dis = self.disease_head(x)

        return anat, dis


model_anat = MultiHeadModel()

model_anat.load_state_dict(ckpt["model_state"])

model_anat.cuda().eval()

print("ANATOMY MODEL LOADED")

In [ ]:
import timm
import torch
import torch.nn as nn

ANATOMY_PATH = "/kaggle/input/models/harshit1631/body-model/pytorch/default/1/best_anatomy_model.pth"

ckpt = torch.load(ANATOMY_PATH, map_location="cpu")

anatomy_cols = ckpt["anatomy_cols"]

num_anatomy = len(anatomy_cols)
num_disease = 15


class GeM(nn.Module):
    def __init__(self, p=3):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)

    def forward(self, x):
        return torch.nn.functional.avg_pool2d(
            x.clamp(min=1e-6).pow(self.p),
            (x.size(-2), x.size(-1))
        ).pow(1. / self.p)


class MultiHeadModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b4",
            pretrained=False,
            num_classes=0
        )

        in_features = self.backbone.num_features

        self.pool = GeM()

        self.shared = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU()
        )

        self.anatomy_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_anatomy)
        )

        self.disease_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_disease)
        )

    def forward(self, x):

        x = self.backbone.forward_features(x)

        x = self.pool(x).flatten(1)

        x = self.shared(x)

        anat = self.anatomy_head(x)
        dis = self.disease_head(x)

        return anat, dis


model_anat = MultiHeadModel()

model_anat.load_state_dict(ckpt["model_state"])

model_anat.cuda().eval()

print("ANATOMY MODEL LOADED")

In [ ]:
import timm
import torch
import torch.nn as nn

ANATOMY_PATH = "/kaggle/input/models/harshit1631/body-model/pytorch/default/1/best_anatomy_model.pth"

ckpt = torch.load(ANATOMY_PATH, map_location="cpu")

anatomy_cols = ckpt["anatomy_cols"]

num_anatomy = len(anatomy_cols)
num_disease = 15


class GeM(nn.Module):
    def __init__(self, p=3):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)

    def forward(self, x):
        return torch.nn.functional.avg_pool2d(
            x.clamp(min=1e-6).pow(self.p),
            (x.size(-2), x.size(-1))
        ).pow(1. / self.p)


class MultiHeadModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b4",
            pretrained=False,
            num_classes=0
        )

        in_features = self.backbone.num_features

        self.pool = GeM()

        # ✅ correct order
        self.shared = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512)
        )

        self.anatomy_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_anatomy)
        )

        self.disease_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_disease)
        )

    def forward(self, x):

        x = self.backbone.forward_features(x)

        x = self.pool(x).flatten(1)

        x = self.shared(x)

        anat = self.anatomy_head(x)
        dis = self.disease_head(x)

        return anat, dis


model_anat = MultiHeadModel()

model_anat.load_state_dict(ckpt["model_state"])

model_anat.cuda().eval()

print("ANATOMY MODEL LOADED")

In [ ]:
model_anat = MultiHeadModel()

model_anat.load_state_dict(
    ckpt["model_state"],
    strict=False
)

model_anat.cuda().eval()

print("ANATOMY MODEL LOADED")

In [ ]:
ALLOWED_LABELS = [
    "mouth",
    "esophagus",
    "stomach",
    "small intestine",
    "colon",
    "z-line",
    "pylorus",
    "ileocecal valve",
    "active bleeding",
    "angiectasia",
    "blood",
    "erosion",
    "erythema",
    "hematin",
    "lymphangioectasis",
    "polyp",
    "ulcer",
]

In [ ]:
DISEASE_LABELS = [
    "active bleeding",
    "angiectasia",
    "blood",
    "erosion",
    "erythema",
    "hematin",
    "lymphangioectasis",
    "polyp",
    "ulcer",
]

In [ ]:
ANATOMY_LABELS = anatomy_cols
print(ANATOMY_LABELS)

In [ ]:
ALLOWED_LABELS = [
    "mouth",
    "esophagus",
    "stomach",
    "small intestine",
    "colon",
    "z-line",
    "pylorus",
    "ileocecal valve",
    "active bleeding",
    "angiectasia",
    "blood",
    "erosion",
    "erythema",
    "hematin",
    "lymphangioectasis",
    "polyp",
    "ulcer",
]


ANATOMY_LABELS = [
    'z-line',
    'pylorus',
    'ampulla of vater',
    'ileocecal valve',
    'section',
    'mouth',
    'esophagus',
    'stomach',
    'small intestine',
    'colon'
]


# mapping anatomy index -> allowed index
anat_to_allowed = {}

for i, name in enumerate(ANATOMY_LABELS):
    if name in ALLOWED_LABELS:
        anat_to_allowed[i] = ALLOWED_LABELS.index(name)

print(anat_to_allowed)

In [ ]:
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as T

device = "cuda"


transform = T.Compose([
    T.Resize((480,480)),
    T.ToTensor(),
])


def predict_frame(img_path):

    img = Image.open(img_path).convert("RGB")
    x = transform(img).unsqueeze(0).to(device)

    # -------- disease ensemble --------
    with torch.no_grad():

        p0 = torch.sigmoid(model0(x))
        p1 = torch.sigmoid(model1(x))
        p2 = torch.sigmoid(model2(x))

        disease = (p0 + p1 + p2) / 3
        disease = disease[0].cpu().numpy()


    # -------- anatomy --------
    with torch.no_grad():

        anat_logits, _ = model_anat(x)
        anat = torch.sigmoid(anat_logits)[0].cpu().numpy()


    # -------- merge to 17 --------

    final = np.zeros(17)


    # anatomy → allowed
    for i, j in anat_to_allowed.items():
        final[j] = anat[i]


    # disease → allowed
    disease_names = [
        "active bleeding",
        "angiectasia",
        "blood",
        "erosion",
        "erythema",
        "hematin",
        "lymphangioectasis",
        "polyp",
        "ulcer",
    ]

    for i, name in enumerate(disease_names):

        if name in ALLOWED_LABELS:

            j = ALLOWED_LABELS.index(name)

            final[j] = disease[i]


    return final

In [ ]:
img = "/kaggle/input/datasets/himanshurajput123/testing/Testdata_ICPR_2026_RARE_Challenge/ukdd_navi_00051/frame_0000049.png"

print(predict_frame(img))

In [ ]:
import os
import pandas as pd
from tqdm import tqdm


def run_video(video_name):

    folder = f"/kaggle/input/datasets/himanshurajput123/testing/Testdata_ICPR_2026_RARE_Challenge/{video_name}"

    frames = sorted(os.listdir(folder))

    rows = []

    for f in tqdm(frames):

        path = os.path.join(folder, f)

        pred = predict_frame(path)

        row = {"frame_file": f}

        for i, name in enumerate(ALLOWED_LABELS):
            row[name] = int(pred[i] > 0.5)

        rows.append(row)


    df = pd.DataFrame(rows)

    out = f"/kaggle/working/{video_name}.csv"

    df.to_csv(out, index=False)

    print("saved", out)

In [ ]:
run_video("ukdd_navi_00051")

In [ ]:
run_video("ukdd_navi_00068")

In [ ]:
run_video("ukdd_navi_00076")

In [ ]:
import os
import shutil

os.makedirs("/kaggle/working/csvs", exist_ok=True)

files = [
    "/kaggle/working/ukdd_navi_00051.csv",
    "/kaggle/working/ukdd_navi_00068.csv",
    "/kaggle/working/ukdd_navi_00076.csv",
]

for f in files:
    shutil.copy(f, "/kaggle/working/csvs")

print("done")

In [ ]:
import pandas as pd
import json
from pathlib import Path


USED_LABELS = [
    "mouth",
    "esophagus",
    "stomach",
    "small intestine",
    "colon",
    "z-line",
    "pylorus",
    "ileocecal valve",
    "active bleeding",
    "angiectasia",
    "blood",
    "erosion",
    "erythema",
    "hematin",
    "lymphangioectasis",
    "polyp",
    "ulcer",
]


def df_to_events(df, video_id):

    events = []

    current = None
    start = 0

    for i, row in df.iterrows():

        labels = tuple(
            sorted(
                [l for l in USED_LABELS if row[l] == 1]
            )
        )

        if current is None:
            current = labels
            start = i

        elif labels != current:

            events.append({
                "start": start,
                "end": i - 1,
                "label": list(current)
            })

            current = labels
            start = i


    events.append({
        "start": start,
        "end": len(df) - 1,
        "label": list(current)
    })

    return {
        "video_id": video_id,
        "events": events
    }


def build_json():

    folder = Path("/kaggle/working/csvs")

    videos = []

    for csv in folder.glob("*.csv"):

        df = pd.read_csv(csv)

        vid = csv.stem

        videos.append(
            df_to_events(df, vid)
        )


    out = {
        "videos": videos
    }

    with open(
        "/kaggle/working/prediction.json",
        "w"
    ) as f:

        json.dump(out, f, indent=2)


build_json()

print("JSON saved")

In [ ]:
import pandas as pd
import json
from pathlib import Path


USED_LABELS = [
    "mouth",
    "esophagus",
    "stomach",
    "small intestine",
    "colon",
    "z-line",
    "pylorus",
    "ileocecal valve",
    "active bleeding",
    "angiectasia",
    "blood",
    "erosion",
    "erythema",
    "hematin",
    "lymphangioectasis",
    "polyp",
    "ulcer",
]


name_map = {
    "ukdd_navi_00051": "vid_001",
    "ukdd_navi_00068": "vid_002",
    "ukdd_navi_00076": "vid_003",
}


def df_to_events(df, video_id):

    events = []

    current = None
    start = 0

    for i, row in df.iterrows():

        labels = tuple(
            sorted(
                [l for l in USED_LABELS if row[l] == 1]
            )
        )

        if current is None:
            current = labels
            start = i

        elif labels != current:

            events.append({
                "start": start,
                "end": i - 1,
                "label": list(current)
            })

            current = labels
            start = i


    events.append({
        "start": start,
        "end": len(df) - 1,
        "label": list(current)
    })

    return {
        "video_id": video_id,
        "events": events
    }


def build_json():

    folder = Path("/kaggle/working/csvs")

    videos = []

    for csv in folder.glob("*.csv"):

        df = pd.read_csv(csv)

        name = csv.stem

        vid = name_map[name]

        videos.append(
            df_to_events(df, vid)
        )


    out = {
        "videos": videos
    }

    with open(
        "/kaggle/working/prediction.json",
        "w"
    ) as f:

        json.dump(out, f, indent=2)


build_json()

print("JSON saved")